In [1]:
%pip install great_expectations

  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/5.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.8 MB ? eta -:--:--
   - ---------------------


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import great_expectations as gx

print("GX version:", gx.__version__)
context = gx.get_context()
print("Context created:", type(context).__name__)

GX version: 1.25.0
Context created: EphemeralDataContext


In [3]:
import os
import pandas as pd

files = [f for f in os.listdir() if f.lower().endswith(".csv") and "superstore" in f.lower()]
FILE = files[0]

try:
    df = pd.read_csv(FILE)
except UnicodeDecodeError:
    df = pd.read_csv(FILE, encoding="latin-1")

df["Order Date"] = pd.to_datetime(df["Order Date"], format="mixed")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="mixed")

print("Loaded:", FILE, "| shape:", df.shape)

Loaded: Sample - Superstore.csv | shape: (9994, 21)


In [4]:
total = len(df)
rows = []

def add(dimension, check, bad, base=total):
    rows.append({
        "Dimension": dimension,
        "Check": check,
        "Bad": int(bad),
        "Checked": int(base),
        "Score_%": round((1 - bad / base) * 100, 2),
    })

# 1. Completeness: no empty cells
add("Completeness", "No empty cells", df.isnull().sum().sum(), df.size)

# 2. Accuracy: values must make sense
add("Accuracy", "Sales is above 0", (df["Sales"] <= 0).sum())
add("Accuracy", "Discount is between 0 and 1", ((df["Discount"] < 0) | (df["Discount"] > 1)).sum())

# 3. Consistency: an order cannot ship before it is ordered
add("Consistency", "Ship Date is not before Order Date", (df["Ship Date"] < df["Order Date"]).sum())

# 4. Uniqueness: no repeated records
add("Uniqueness", "Row ID is unique", df["Row ID"].duplicated().sum())
add("Uniqueness", "No fully duplicate rows", df.duplicated().sum())

# 5. Validity: values follow the expected format
add("Validity", "Order ID has the format XX-YYYY-number", (~df["Order ID"].str.match(r"^[A-Z]{2}-\d{4}-\d+$")).sum())
add("Validity", "Segment is an allowed value", (~df["Segment"].isin(["Consumer", "Corporate", "Home Office"])).sum())

scorecard = pd.DataFrame(rows)
scorecard

,Dimension,Check,Bad,Checked,Score_%
0,Completeness,No empty cells,0,209874,100.0
1,Accuracy,Sales is above 0,0,9994,100.0
2,Accuracy,Discount is between 0 and 1,0,9994,100.0
3,Consistency,Ship Date is not before Order Date,0,9994,100.0
4,Uniqueness,Row ID is unique,0,9994,100.0
5,Uniqueness,No fully duplicate rows,0,9994,100.0
6,Validity,Order ID has the format XX-YYYY-number,0,9994,100.0
7,Validity,Segment is an allowed value,0,9994,100.0


In [5]:
latest = df["Order Date"].max()
age_days = (pd.Timestamp.today() - latest).days
print("Latest order date:", latest.date())
print("Data is", age_days, "days old")

Latest order date: 2017-12-30
Data is 3248 days old


In [6]:
import great_expectations.expectations as gxe

suite = gx.ExpectationSuite(name="superstore_quality_suite")

rules = [
    # Completeness
    gxe.ExpectColumnValuesToNotBeNull(column="Order ID"),
    # Uniqueness
    gxe.ExpectColumnValuesToBeUnique(column="Row ID"),
    # Accuracy
    gxe.ExpectColumnValuesToBeBetween(column="Sales", min_value=0, strict_min=True),
    gxe.ExpectColumnValuesToBeBetween(column="Discount", min_value=0, max_value=1),
    # Validity
    gxe.ExpectColumnValuesToBeInSet(column="Segment", value_set=["Consumer", "Corporate", "Home Office"]),
    gxe.ExpectColumnValuesToBeInSet(column="Ship Mode", value_set=["First Class", "Second Class", "Standard Class", "Same Day"]),
    gxe.ExpectColumnValuesToMatchRegex(column="Order ID", regex=r"^[A-Z]{2}-\d{4}-\d+$"),
    # Consistency
    gxe.ExpectColumnPairValuesAToBeGreaterThanB(column_A="Ship Date", column_B="Order Date", or_equal=True),
]

for rule in rules:
    suite.add_expectation(rule)

context.suites.add(suite)
print("Suite created with", len(suite.expectations), "expectations")

Suite created with 8 expectations


In [7]:
data_source = context.data_sources.add_pandas(name="superstore_source")
data_asset = data_source.add_dataframe_asset(name="superstore_asset")
batch_definition = data_asset.add_batch_definition_whole_dataframe("whole_dataframe")

batch = batch_definition.get_batch(batch_parameters={"dataframe": df})
print("Batch ready")

Batch ready


In [8]:
result = batch.validate(suite)

print("Overall success:", result.success)
print(result.statistics)

for r in result.results:
    print(r.success, "|", r.expectation_config.type)

Calculating Metrics: 100%|██████████| 54/54 [00:00<00:00, 510.64it/s]

Overall success: True
{'evaluated_expectations': 8, 'successful_expectations': 8, 'unsuccessful_expectations': 0, 'success_percent': 100.0}
True | expect_column_values_to_not_be_null
True | expect_column_values_to_match_regex
True | expect_column_values_to_be_unique
True | expect_column_values_to_be_between
True | expect_column_values_to_be_between
True | expect_column_values_to_be_in_set
True | expect_column_values_to_be_in_set
True | expect_column_pair_values_a_to_be_greater_than_b
